In [0]:
from pyspark.sql import functions as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

gold_delivery_performance_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/delivery_performance/"
)

silver_carriers_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/carriers/"
)

gold_carrier_performance_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "gold/carrier_performance/"
)


# ============================================================
# 2. LECTURE
# ============================================================

delivery_performance = (
    spark.read
    .format("delta")
    .load(gold_delivery_performance_path)
)

silver_carriers = (
    spark.read
    .format("delta")
    .load(silver_carriers_path)
)


# ============================================================
# 3. AGRÉGATION PAR CARRIER
# ============================================================

carrier_metrics = (
    delivery_performance

    .groupBy("carrier_id")

    .agg(

        # Nombre total de shipments
        F.count("*")
        .alias("total_shipments"),

        # Nombre de shipments livrés
        F.sum(
            F.col("is_delivered").cast("int")
        )
        .alias("delivered_shipments"),

        # Nombre de shipments en retard
        F.sum(
            F.when(
                F.col("is_late") == True,
                1
            ).otherwise(0)
        )
        .alias("late_shipments"),

        # Nombre de shipments livrés à l'heure
        F.sum(
            F.when(
                (F.col("is_delivered") == True)
                & (F.col("is_late") == False),
                1
            ).otherwise(0)
        )
        .alias("on_time_shipments"),

        # Durée moyenne de livraison
        F.avg(
            "delivery_duration_hours"
        )
        .alias("avg_delivery_duration_hours"),

        # Retard moyen
        F.avg(
            F.when(
                F.col("is_late") == True,
                F.col("delivery_delay_hours")
            )
        )
        .alias("avg_late_delay_hours")
    )
)


# ============================================================
# 4. CALCUL DU TAUX DE LIVRAISON À L'HEURE
# ============================================================

carrier_metrics = (
    carrier_metrics

    .withColumn(
        "on_time_delivery_rate_pct",

        F.when(
            F.col("delivered_shipments") > 0,

            F.round(
                (
                    F.col("on_time_shipments")
                    / F.col("delivered_shipments")
                ) * 100,
                2
            )
        )

        .otherwise(
            F.lit(None)
        )
    )
)


# ============================================================
# 5. AJOUTER LES INFORMATIONS DU CARRIER
# ============================================================

carrier_reference = (
    silver_carriers
    .select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "country",
        "service_level",
        "active"
    )
)

gold_carrier_performance = (
    carrier_metrics

    .join(
        carrier_reference,
        on="carrier_id",
        how="left"
    )

    .select(
        "carrier_id",
        "carrier_name",
        "carrier_type",
        "country",
        "service_level",
        "active",
        "total_shipments",
        "delivered_shipments",
        "late_shipments",
        "on_time_shipments",
        "avg_delivery_duration_hours",
        "avg_late_delay_hours",
        "on_time_delivery_rate_pct"
    )
)


# ============================================================
# 6. ÉCRITURE GOLD
# ============================================================

(
    gold_carrier_performance
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_carrier_performance_path)
)

print(
    "gold_carrier_performance créée avec succès."
)


# ============================================================
# 7. AFFICHAGE
# ============================================================

display(
    gold_carrier_performance
    .orderBy("carrier_id")
)